## Cell 1 — Setup + paths + retrieval settings

In [8]:
# ============================================================
# Cell 1 — Setup + paths + retrieval settings
# Run once per session
# ============================================================

!pip install -q sentence-transformers pandas==2.2.2

from google.colab import drive
drive.mount("/content/drive")

import os
import re
import csv
import json
import gc
from datetime import datetime
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import torch

# ============================================================
# Paths
# ============================================================

BASE_DIR = "/content/drive/MyDrive/Islamic_Stories_Project"

# Hadith corpus (BGE embeddings already prepared)
HADITH_META_CSV  = f"{BASE_DIR}/hadith_embedding/hadith_meta_bge.csv"
HADITH_EMBEDS_PT = f"{BASE_DIR}/hadith_embedding/hadith_embeds_bge.pt"

# Per-model input / output files
MODELS = {
    "ALLaM": {
        "stories_csv": f"{BASE_DIR}/ALLaM_output.csv",
        "output_csv":  f"{BASE_DIR}/ALLaM_retriever_results.csv",
    },
    "LlaMa": {
        "stories_csv": f"{BASE_DIR}/LlaMa_output.csv",
        "output_csv":  f"{BASE_DIR}/LlaMa_retriever_results.csv",
    },
    "Qwen": {
        "stories_csv": f"{BASE_DIR}/Qwen_output.csv",
        "output_csv":  f"{BASE_DIR}/Qwen_retriever_results.csv",
    },
}

# ============================================================
# Retrieval settings — identical to the original notebook
# ============================================================

INITIAL_K              = 50
FINAL_K                = 10
SIMILARITY_THRESHOLD   = 0.78
CONTAINMENT_THRESHOLD  = 0.75

# If True, the per-model output CSV is overwritten when you re-run that cell.
# Set False if you want to keep appending.
OVERWRITE_OUTPUT = True

# ============================================================
# Sanity check
# ============================================================

print("Hadith corpus files:")
for label, p in [("meta", HADITH_META_CSV), ("embeddings", HADITH_EMBEDS_PT)]:
    print(("  ✓" if os.path.exists(p) else "  ✗ MISSING"), label, "->", p)

print()
print("Per-model files:")
for name, cfg in MODELS.items():
    exists = os.path.exists(cfg["stories_csv"])
    print(f"  {name}")
    print(f"    stories: {'✓' if exists else '✗ MISSING'}  {cfg['stories_csv']}")
    print(f"    output : {cfg['output_csv']}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hadith corpus files:
  ✓ meta -> /content/drive/MyDrive/Islamic_Stories_Project/hadith_embedding/hadith_meta_bge.csv
  ✓ embeddings -> /content/drive/MyDrive/Islamic_Stories_Project/hadith_embedding/hadith_embeds_bge.pt

Per-model files:
  ALLaM
    stories: ✓  /content/drive/MyDrive/Islamic_Stories_Project/ALLaM_output.csv
    output : /content/drive/MyDrive/Islamic_Stories_Project/ALLaM_retriever_results.csv
  LlaMa
    stories: ✓  /content/drive/MyDrive/Islamic_Stories_Project/LlaMa_output.csv
    output : /content/drive/MyDrive/Islamic_Stories_Project/LlaMa_retriever_results.csv
  Qwen
    stories: ✓  /content/drive/MyDrive/Islamic_Stories_Project/Qwen_output.csv
    output : /content/drive/MyDrive/Islamic_Stories_Project/Qwen_retriever_results.csv


## Cell 2 — Load Hadith corpus + BGE-M3 + CrossEncoder

Same as the original pipeline. Both retrieval models run on GPU.

In [4]:
# ============================================================
# Cell 2 — Load Hadith corpus + retrieval models
# ============================================================

from sentence_transformers import SentenceTransformer, CrossEncoder, util

# Hadith corpus
hadith_df = pd.read_csv(HADITH_META_CSV)
hadith_texts = hadith_df["text_ar"].astype(str).tolist()
hadith_embeds = torch.load(HADITH_EMBEDS_PT, map_location="cuda")
print(f"✓ Hadith corpus loaded: {len(hadith_df)} entries")

# BGE-M3 encoder
embed_model = SentenceTransformer("BAAI/bge-m3").to("cuda")
print("✓ BGE-M3 loaded")

# CrossEncoder reranker
reranker = CrossEncoder("BAAI/bge-reranker-base", device="cuda")
print("✓ bge-reranker-base loaded")

print("\nAll retrieval resources ready.")


✓ Hadith corpus loaded: 34433 entries


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

✓ bge-reranker-base loaded

All retrieval resources ready.


## Cell 3 — Helpers (Arabic normalization, matn extraction, dedup)

Copied verbatim from the original notebook. Arabic normalization is applied for dedup only — it does **not** change the Hadith text shown in results.

In [9]:
# ============================================================
# Cell 3 — Helpers (identical to the original pipeline)
# ============================================================

# ------------------------------------------------------------
# JSON-safe helper — fixes: TypeError: Object of type int64 is not JSON serializable
# ------------------------------------------------------------

def make_json_safe(obj):
    """Convert numpy / pandas values into plain Python types for JSON/CSV saving."""
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return tuple(make_json_safe(v) for v in obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


def safe_get(row, col, default=""):
    """Read a value from a CSV row, returning default for missing/NaN."""
    if col not in row:
        return default
    val = row.get(col, default)
    try:
        if pd.isna(val):
            return default
    except Exception:
        pass
    return val


# ------------------------------------------------------------
# Arabic normalization — for duplicate detection ONLY
# ------------------------------------------------------------

def normalize_arabic(text):
    """Normalize Arabic text to make duplicate detection easier."""
    if text is None:
        return ""

    text = str(text)

    # Remove Arabic diacritics
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652]", "", text)

    # Remove tatweel
    text = text.replace("ـ", "")

    # Normalize common Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ة", "ه")
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")

    # Remove punctuation and symbols
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)

    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_spaces(text):
    if text is None:
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def extract_hadith_matn(text):
    """
    Try to extract the quoted Hadith matn from the full Hadith text.
    If the Hadith contains quoted text, use the longest quoted segment.
    Otherwise, use the full text.
    """
    if text is None:
        return ""

    text = str(text)

    patterns = [
        r"‏\"‏(.*?)‏\"‏",
        r'"(.*?)"',
        r"“(.*?)”",
        r"«(.*?)»",
    ]

    matches = []
    for pat in patterns:
        found = re.findall(pat, text, flags=re.DOTALL)
        matches.extend(found)

    if matches:
        matn = max(matches, key=len)
        return clean_spaces(matn)

    return clean_spaces(text)


def candidate_dedup_key(candidate):
    """
    Build comparison text for duplicate detection.
    Prefer matn if extractable (and long enough); otherwise use full Hadith text.
    Arabic normalization is applied here only.
    """
    text = candidate.get("text_ar", "")
    matn = extract_hadith_matn(text)

    if len(matn.split()) < 4:
        matn = clean_spaces(text)

    return normalize_arabic(matn)


# ------------------------------------------------------------
# Deduplication AFTER CrossEncoder reranking
# ------------------------------------------------------------

def deduplicate_after_crossencoder(
    candidates,
    similarity_threshold=0.78,
    containment_threshold=0.75,
    final_k=10,
):
    """
    Remove near-duplicate Hadith candidates AFTER CrossEncoder reranking.

    A candidate is removed if either:
    1. Sequence similarity with an already-kept Hadith is high.
    2. Most of its words overlap with an already-kept Hadith.

    Doing dedup after CE means we keep the best-ranked version of each narration.
    """
    unique = []
    unique_keys = []

    for cand in candidates:
        cand_key = candidate_dedup_key(cand)

        if not cand_key:
            continue

        cand_words = set(cand_key.split())
        is_duplicate = False

        for kept_key in unique_keys:
            kept_words = set(kept_key.split())

            seq_sim = SequenceMatcher(None, cand_key, kept_key).ratio()

            if len(cand_words) == 0 or len(kept_words) == 0:
                containment = 0
            else:
                overlap = len(cand_words.intersection(kept_words))
                containment = overlap / min(len(cand_words), len(kept_words))

            if seq_sim >= similarity_threshold or containment >= containment_threshold:
                is_duplicate = True
                break

        if not is_duplicate:
            unique.append(cand)
            unique_keys.append(cand_key)

        if len(unique) >= final_k:
            break

    for rank, cand in enumerate(unique, start=1):
        cand["rank_after_dedup"] = rank

    return unique


print("✓ Helpers ready")


✓ Helpers ready


## Cell 4 — Retriever, CSV writer, and per-model runner

`run_retrieval_for_model(model_name)` reads the stories CSV for a model, runs retrieval **twice** per row (with and without summary), and writes one results CSV for that model.

In [10]:
# ============================================================
# Cell 4 — Retriever + CSV writer + per-model runner
# ============================================================

def retrieve_bge_ce_dedup(
    retrieval_query,
    initial_k=INITIAL_K,
    final_k=FINAL_K,
    similarity_threshold=SIMILARITY_THRESHOLD,
    containment_threshold=CONTAINMENT_THRESHOLD,
):
    """
    BGE top-N -> CrossEncoder rerank -> dedup -> top-K
    """

    # -------- BGE retrieval --------
    q_emb = embed_model.encode(
        retrieval_query,
        convert_to_tensor=True,
        normalize_embeddings=True,
        device="cuda",
    )

    sims = util.cos_sim(q_emb, hadith_embeds)[0].cuda().tolist()

    top_idx = sorted(
        range(len(sims)),
        key=lambda i: sims[i],
        reverse=True,
    )[:initial_k]

    candidates = []
    for bge_rank, i in enumerate(top_idx, start=1):
        row_h = hadith_df.iloc[i]
        candidates.append({
            "idx":       int(i),
            "rank_bge":  bge_rank,
            "score_bge": float(sims[i]),
            "text_ar":   str(hadith_texts[i]),
            "source":    make_json_safe(row_h.get("source")),
            "chapter":   make_json_safe(row_h.get("chapter")),
            "hadith_id": make_json_safe(row_h.get("hadith_id")),
        })

    # -------- CrossEncoder reranking --------
    if candidates:
        pairs = [(retrieval_query, c["text_ar"]) for c in candidates]
        ce_scores = reranker.predict(pairs).tolist()

        for c, s in zip(candidates, ce_scores):
            c["score_ce"] = float(s)

        reranked = sorted(candidates, key=lambda c: c["score_ce"], reverse=True)

        for rank, c in enumerate(reranked, start=1):
            c["rank_after_ce"] = rank
    else:
        reranked = []

    # -------- Dedup AFTER CrossEncoder --------
    final_candidates = deduplicate_after_crossencoder(
        reranked,
        similarity_threshold=similarity_threshold,
        containment_threshold=containment_threshold,
        final_k=final_k,
    )

    selected = final_candidates[0] if final_candidates else None

    return {
        "initial_k": initial_k,
        "final_k": final_k,
        "similarity_threshold": similarity_threshold,
        "containment_threshold": containment_threshold,
        "bge_candidates":      candidates,
        "reranked_candidates": reranked,
        "final_candidates":    final_candidates,
        "selected":            selected,
    }


# ============================================================
# CSV columns
# ============================================================

RESULT_COLUMNS = [
    "sample_id",
    "timestamp",
    "model_name",

    # Carried over from the stories CSV
    "input_age",
    "input_topic",
    "input_moral",
    "input_place",
    "input_country",
    "input_season",
    "input_activity",
    "input_emotion",
    "input_dialogue",
    "input_plot_twist",
    "input_end_of_story",
    "generated_story",
    "extracted_moral",

    # Retrieval settings (for reproducibility)
    "initial_k",
    "final_k",
    "dedup_similarity_threshold",
    "dedup_containment_threshold",

    # WITH-summary retrieval
    "with_summary_query",
    "with_summary_top_k_candidates",
    "with_summary_selected_hadith",
    "with_summary_hadith_source",
    "with_summary_hadith_chapter",
    "with_summary_hadith_id",
    "with_summary_bge_score",
    "with_summary_cross_encoder_score",

    # WITHOUT-summary retrieval
    "no_summary_query",
    "no_summary_top_k_candidates",
    "no_summary_selected_hadith",
    "no_summary_hadith_source",
    "no_summary_hadith_chapter",
    "no_summary_hadith_id",
    "no_summary_bge_score",
    "no_summary_cross_encoder_score",
]


def _serialize_candidates(final_candidates):
    safe = make_json_safe([
        {
            "rank_after_dedup": c.get("rank_after_dedup"),
            "rank_after_ce":    c.get("rank_after_ce"),
            "rank_bge":         c.get("rank_bge"),
            "idx":              c.get("idx"),
            "text_ar":          c.get("text_ar"),
            "source":           c.get("source"),
            "chapter":          c.get("chapter"),
            "hadith_id":        c.get("hadith_id"),
            "score_bge":        c.get("score_bge"),
            "score_ce":         c.get("score_ce"),
        }
        for c in final_candidates
    ])
    return json.dumps(safe, ensure_ascii=False)


def _csv_has_header(path):
    if not os.path.exists(path):
        return False
    try:
        with open(path, "r", encoding="utf-8-sig", newline="") as f:
            first = f.readline()
        return first.strip().startswith("sample_id")
    except Exception:
        return False


def append_result_row(path, source_row, sample_id, model_name, with_result, no_result):
    """Save one row containing both retrieval conditions for this model."""

    parent = os.path.dirname(os.path.abspath(path))
    if parent:
        os.makedirs(parent, exist_ok=True)

    write_header = not _csv_has_header(path)
    mode = "a" if os.path.exists(path) else "w"

    with_sel = with_result.get("selected") or {}
    no_sel   = no_result.get("selected")   or {}

    row = {
        "sample_id":  sample_id,
        "timestamp":  datetime.now().isoformat(timespec="seconds"),
        "model_name": model_name,

        "input_age":          safe_get(source_row, "input_age"),
        "input_topic":        safe_get(source_row, "input_topic"),
        "input_moral":        safe_get(source_row, "input_moral"),
        "input_place":        safe_get(source_row, "input_place"),
        "input_country":      safe_get(source_row, "input_country"),
        "input_season":       safe_get(source_row, "input_season"),
        "input_activity":     safe_get(source_row, "input_activity"),
        "input_emotion":      safe_get(source_row, "input_emotion"),
        "input_dialogue":     safe_get(source_row, "input_dialogue"),
        "input_plot_twist":   safe_get(source_row, "input_plot_twist"),
        "input_end_of_story": safe_get(source_row, "input_end_of_story"),
        "generated_story":    safe_get(source_row, "generated_story"),
        "extracted_moral":    safe_get(source_row, "extracted_moral"),

        "initial_k":                   with_result.get("initial_k"),
        "final_k":                     with_result.get("final_k"),
        "dedup_similarity_threshold":  with_result.get("similarity_threshold"),
        "dedup_containment_threshold": with_result.get("containment_threshold"),

        # WITH summary
        "with_summary_query":               safe_get(source_row, "query_with_summary"),
        "with_summary_top_k_candidates":    _serialize_candidates(with_result.get("final_candidates") or []),
        "with_summary_selected_hadith":     with_sel.get("text_ar", ""),
        "with_summary_hadith_source":       with_sel.get("source", ""),
        "with_summary_hadith_chapter":      with_sel.get("chapter", ""),
        "with_summary_hadith_id":           with_sel.get("hadith_id", ""),
        "with_summary_bge_score":           with_sel.get("score_bge", ""),
        "with_summary_cross_encoder_score": with_sel.get("score_ce", ""),

        # WITHOUT summary
        "no_summary_query":               safe_get(source_row, "query_no_summary"),
        "no_summary_top_k_candidates":    _serialize_candidates(no_result.get("final_candidates") or []),
        "no_summary_selected_hadith":     no_sel.get("text_ar", ""),
        "no_summary_hadith_source":       no_sel.get("source", ""),
        "no_summary_hadith_chapter":      no_sel.get("chapter", ""),
        "no_summary_hadith_id":           no_sel.get("hadith_id", ""),
        "no_summary_bge_score":           no_sel.get("score_bge", ""),
        "no_summary_cross_encoder_score": no_sel.get("score_ce", ""),
    }

    with open(path, mode, encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RESULT_COLUMNS)
        if write_header:
            writer.writeheader()
        writer.writerow(row)

    return row


# ============================================================
# Per-model runner
# ============================================================

def run_retrieval_for_model(model_name):
    """
    Read the stories CSV for `model_name`, run retrieval twice per row
    (WITH summary and WITHOUT summary), and save results to a per-model CSV.
    """
    if model_name not in MODELS:
        raise ValueError(f"Unknown model: {model_name}. Expected one of: {list(MODELS)}")

    cfg = MODELS[model_name]
    stories_path = cfg["stories_csv"]
    output_path  = cfg["output_csv"]

    print("=" * 80)
    print(f"Running retrieval for model: {model_name}")
    print(f"  stories CSV: {stories_path}")
    print(f"  output  CSV: {output_path}")
    print("=" * 80)

    if not os.path.exists(stories_path):
        raise FileNotFoundError(f"Stories CSV not found: {stories_path}")

    if OVERWRITE_OUTPUT and os.path.exists(output_path):
        os.remove(output_path)
        print("Old output deleted. Starting fresh.\n")

    stories_df = pd.read_csv(stories_path)
    print(f"Rows to process: {len(stories_df)}")

    for idx, source_row in stories_df.iterrows():
        sample_id = safe_get(source_row, "sample_id", idx + 1)

        topic = safe_get(source_row, "input_topic")
        moral = safe_get(source_row, "input_moral")

        print("\n" + "-" * 80)
        print(f"[{model_name}] sample {sample_id}  topic={topic}  moral={moral}")
        print("-" * 80)

        # --- WITH summary ---
        with_query = safe_get(source_row, "query_with_summary")
        print("\n[WITH summary] query:")
        print(with_query)
        with_result = retrieve_bge_ce_dedup(with_query)
        ws = with_result.get("selected") or {}
        print(f"[WITH summary] selected hadith_id={ws.get('hadith_id')}  "
              f"BGE={ws.get('score_bge', 0):.4f}  CE={ws.get('score_ce', 0):.4f}")

        # --- WITHOUT summary ---
        no_query = safe_get(source_row, "query_no_summary")
        print("\n[NO summary] query:")
        print(no_query)
        no_result = retrieve_bge_ce_dedup(no_query)
        ns = no_result.get("selected") or {}
        print(f"[NO  summary] selected hadith_id={ns.get('hadith_id')}  "
              f"BGE={ns.get('score_bge', 0):.4f}  CE={ns.get('score_ce', 0):.4f}")

        # --- Save ---
        append_result_row(
            path=output_path,
            source_row=source_row,
            sample_id=sample_id,
            model_name=model_name,
            with_result=with_result,
            no_result=no_result,
        )
        print("  ✓ saved")

    print("\n" + "=" * 80)
    print(f"Done. Saved {len(stories_df)} rows to:")
    print(output_path)
    print("=" * 80)


print("✓ Retriever, writer, and per-model runner ready")


✓ Retriever, writer, and per-model runner ready


## Cell 5 — Run retrieval on ALLaM stories

In [11]:
run_retrieval_for_model("ALLaM")


Running retrieval for model: ALLaM
  stories CSV: /content/drive/MyDrive/Islamic_Stories_Project/ALLaM_output.csv
  output  CSV: /content/drive/MyDrive/Islamic_Stories_Project/ALLaM_retriever_results.csv
Rows to process: 3

--------------------------------------------------------------------------------
[ALLaM] sample 1  topic=الصدق  moral=أهمية الصدق
--------------------------------------------------------------------------------

[WITH summary] query:
القيمة الإسلامية: الصدق
العبرة من القصة: أهمية الصدق
ملخص القصة: الصدق يحمي ويوحد الأصدقاء.
[WITH summary] selected hadith_id=16704  BGE=0.6118  CE=0.7166

[NO summary] query:
القيمة الإسلامية: الصدق
العبرة من القصة: أهمية الصدق
[NO  summary] selected hadith_id=23957  BGE=0.5429  CE=0.7868
  ✓ saved

--------------------------------------------------------------------------------
[ALLaM] sample 2  topic=الصيام  moral=فضل الصيام
--------------------------------------------------------------------------------

[WITH summary] query:
القيمة

## Cell 6 — Run retrieval on LlaMa stories

In [12]:
run_retrieval_for_model("LlaMa")


Running retrieval for model: LlaMa
  stories CSV: /content/drive/MyDrive/Islamic_Stories_Project/LlaMa_output.csv
  output  CSV: /content/drive/MyDrive/Islamic_Stories_Project/LlaMa_retriever_results.csv
Rows to process: 3

--------------------------------------------------------------------------------
[LlaMa] sample 1  topic=الصدق  moral=أهمية الصدق
--------------------------------------------------------------------------------

[WITH summary] query:
القيمة الإسلامية: الصدق
العبرة من القصة: أهمية الصدق
ملخص القصة: في اللعبة، اكتشف سلمان أن الصدق والتعاون يفيدان أكثر من الفوز وحده.
[WITH summary] selected hadith_id=16704  BGE=0.5587  CE=0.2267

[NO summary] query:
القيمة الإسلامية: الصدق
العبرة من القصة: أهمية الصدق
[NO  summary] selected hadith_id=23957  BGE=0.5429  CE=0.7868
  ✓ saved

--------------------------------------------------------------------------------
[LlaMa] sample 2  topic=الصيام  moral=فضل الصيام
---------------------------------------------------------------------

## Cell 7 — Run retrieval on Qwen stories

In [13]:
run_retrieval_for_model("Qwen")


Running retrieval for model: Qwen
  stories CSV: /content/drive/MyDrive/Islamic_Stories_Project/Qwen_output.csv
  output  CSV: /content/drive/MyDrive/Islamic_Stories_Project/Qwen_retriever_results.csv
Rows to process: 3

--------------------------------------------------------------------------------
[Qwen] sample 1  topic=الصدق  moral=أهمية الصدق
--------------------------------------------------------------------------------

[WITH summary] query:
القيمة الإسلامية: الصدق
العبرة من القصة: أهمية الصدق
ملخص القصة: جوهرة صدقت بأنها لم تستخدم قلمها الجديد مما أظهر أهمية الصدق وحفظ الثقة، لكنها لاحظت في النهاية أن الكلمة "صدق" لم تكتمل مما أضاف درسًا إضافيًا حول الدقة في الصدق.
[WITH summary] selected hadith_id=42549  BGE=0.5534  CE=0.4526

[NO summary] query:
القيمة الإسلامية: الصدق
العبرة من القصة: أهمية الصدق
[NO  summary] selected hadith_id=23957  BGE=0.5429  CE=0.7868
  ✓ saved

--------------------------------------------------------------------------------
[Qwen] sample 2  topic=الص

## Cell 8 — Quick comparison: each model's chosen Hadith per sample

Loads all three result CSVs and shows a compact comparison table per model.

In [14]:
from IPython.display import display

for model_name, cfg in MODELS.items():
    path = cfg["output_csv"]
    if not os.path.exists(path):
        print(f"⚠ {model_name}: no results yet ({path})")
        continue

    df = pd.read_csv(path)
    print("=" * 80)
    print(f"{model_name}  —  {len(df)} rows  —  {path}")
    print("=" * 80)

    compact = df[[
        "sample_id",
        "input_topic",
        "input_moral",
        "with_summary_hadith_id",
        "with_summary_cross_encoder_score",
        "no_summary_hadith_id",
        "no_summary_cross_encoder_score",
    ]].copy()

    compact["same_hadith"] = (
        compact["with_summary_hadith_id"].astype(str)
        == compact["no_summary_hadith_id"].astype(str)
    )
    display(compact)
    print(f"Same top hadith picked under both conditions: "
          f"{compact['same_hadith'].sum()}/{len(compact)}")
    print()


ALLaM  —  3 rows  —  /content/drive/MyDrive/Islamic_Stories_Project/ALLaM_retriever_results.csv


,sample_id,input_topic,input_moral,with_summary_hadith_id,with_summary_cross_encoder_score,no_summary_hadith_id,no_summary_cross_encoder_score,same_hadith
0,1,الصدق,أهمية الصدق,16704,0.716612,23957,0.786828,False
1,2,الصيام,فضل الصيام,6537,0.287372,30808,0.489729,False
2,3,الصلاة,أهمية الصلاة,53970,0.822600,7382,0.387312,False


Same top hadith picked under both conditions: 0/3

LlaMa  —  3 rows  —  /content/drive/MyDrive/Islamic_Stories_Project/LlaMa_retriever_results.csv


,sample_id,input_topic,input_moral,with_summary_hadith_id,with_summary_cross_encoder_score,no_summary_hadith_id,no_summary_cross_encoder_score,same_hadith
0,1,الصدق,أهمية الصدق,16704,0.226744,23957,0.786828,False
1,2,الصيام,فضل الصيام,42406,0.184552,30808,0.489729,False
2,3,الصلاة,أهمية الصلاة,53970,0.690702,7382,0.387312,False


Same top hadith picked under both conditions: 0/3

Qwen  —  3 rows  —  /content/drive/MyDrive/Islamic_Stories_Project/Qwen_retriever_results.csv


,sample_id,input_topic,input_moral,with_summary_hadith_id,with_summary_cross_encoder_score,no_summary_hadith_id,no_summary_cross_encoder_score,same_hadith
0,1,الصدق,أهمية الصدق,42549,0.452550,23957,0.786828,False
1,2,الصيام,فضل الصيام,51822,0.251999,30808,0.489729,False
2,3,الصلاة,أهمية الصلاة,20763,0.578162,7382,0.387312,False


Same top hadith picked under both conditions: 0/3



## Cell 9 — Inspect one sample in detail

Set `MODEL_NAME` and `SAMPLE_ID` to see the full top-10 list (both conditions) for a specific story.

In [15]:
MODEL_NAME = "ALLaM"   # "ALLaM" | "LlaMa" | "Qwen"
SAMPLE_ID  = 1

path = MODELS[MODEL_NAME]["output_csv"]
df = pd.read_csv(path)
row = df[df["sample_id"] == SAMPLE_ID].iloc[0]

print("=" * 80)
print(f"Model : {MODEL_NAME}")
print(f"Sample: {SAMPLE_ID}")
print("=" * 80)
print("Topic :", row["input_topic"])
print("Moral :", row["input_moral"])
print("Story extracted moral:", row["extracted_moral"])

for label, q_col, top_col in [
    ("WITH summary", "with_summary_query",   "with_summary_top_k_candidates"),
    ("NO  summary",  "no_summary_query",     "no_summary_top_k_candidates"),
]:
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)
    print("Query:")
    print(row[q_col])
    print()
    cands = json.loads(row[top_col])
    print(f"Top-{len(cands)} after CrossEncoder + dedup:")
    for c in cands:
        print("\n" + "-" * 60)
        print(f"  rank_after_dedup : {c.get('rank_after_dedup')}")
        print(f"  rank_after_ce    : {c.get('rank_after_ce')}")
        print(f"  rank_bge         : {c.get('rank_bge')}")
        print(f"  hadith_id        : {c.get('hadith_id')}")
        print(f"  source           : {c.get('source')}")
        print(f"  chapter          : {c.get('chapter')}")
        print(f"  BGE score        : {c.get('score_bge', 0):.4f}")
        print(f"  CE  score        : {c.get('score_ce', 0):.4f}")
        print("  text:")
        print(" " * 4 + str(c.get("text_ar", ""))[:500])


Model : ALLaM
Sample: 1
Topic : الصدق
Moral : أهمية الصدق
Story extracted moral: الصدق هو ما يجعلنا أقوى ويحمي من حولنا.

WITH summary
Query:
القيمة الإسلامية: الصدق
العبرة من القصة: أهمية الصدق
ملخص القصة: الصدق يحمي ويوحد الأصدقاء.

Top-10 after CrossEncoder + dedup:

------------------------------------------------------------
  rank_after_dedup : 1
  rank_after_ce    : 1
  rank_bge         : 2
  hadith_id        : 16704
  source           :  Sahih Muslim 
  chapter          : The Book of Virtue; Good Manners and Joining of the Ties of Relationship - كتاب البر والصلة والآداب
  BGE score        : 0.6118
  CE  score        : 0.7166
  text:
    حدثنا محمد بن عبد الله بن نمير، حدثنا أبو معاوية، ووكيع، قالا حدثنا الأعمش، ح وحدثنا أبو كريب، حدثنا أبو معاوية، حدثنا الأعمش، عن شقيق، عن عبد الله، قال قال رسول الله صلى الله عليه وسلم ‏"‏ عليكم بالصدق فإن الصدق يهدي إلى البر وإن البر يهدي إلى الجنة وما يزال الرجل يصدق ويتحرى الصدق حتى يكتب عند الله صديقا وإياكم والكذب فإن الكذب يهدي إلى الفجور